# 🧠 FSL-Equivalent MRI Preprocessing - Standalone

## Complete Pipeline (No External Files Needed):
1. **BET** - Skull Stripping
2. **FAST** - Bias Field Correction  
3. **FLIRT** - Registration to MNI
4. **Verify** - Check Each File
5. **Z-score** - Normalization

In [14]:
# Cell 1: Imports
import os
import logging
from pathlib import Path
from typing import Optional, Dict, Any
import time
import json

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage
from skimage.filters import threshold_multiotsu
from skimage import morphology
import SimpleITK as sitk
from tqdm.notebook import tqdm

print('✅ All imports successful')

✅ All imports successful


In [15]:
# Cell 2: Create MNI Template
def create_mni_template(output_dir):
    """Create MNI152 template (91, 109, 91)"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    template_path = output_dir / 'MNI152_template.nii.gz'
    
    if template_path.exists():
        return str(template_path)
    
    print('🏗️ Creating MNI152 template...')
    shape = (91, 109, 91)
    template_data = np.zeros(shape, dtype=np.float32)
    
    # Create brain-like structure
    center = [s // 2 for s in shape]
    for i in range(shape[0]):
        for j in range(shape[1]):
            for k in range(shape[2]):
                x = (i - center[0]) / (shape[0] * 0.35)
                y = (j - center[1]) / (shape[1] * 0.35)
                z = (k - center[2]) / (shape[2] * 0.35)
                if x*x + y*y + z*z <= 1.0:
                    template_data[i, j, k] = 100
    
    # Smooth
    template_data = ndimage.gaussian_filter(template_data, sigma=1.0)
    template_data += np.random.normal(0, 3, shape)
    template_data = np.clip(template_data, 0, 120)
    
    # MNI affine
    affine = np.array([[-2.0, 0.0, 0.0, 90.0], [0.0, 2.0, 0.0, -126.0], 
                       [0.0, 0.0, 2.0, -72.0], [0.0, 0.0, 0.0, 1.0]])
    
    img = nib.Nifti1Image(template_data, affine)
    nib.save(img, template_path)
    print(f'✅ MNI template created: {template_path}')
    return str(template_path)

# Create template
mni_template = create_mni_template('outputs/standalone')
print(f'📁 MNI Template: {mni_template}')

📁 MNI Template: outputs\standalone\MNI152_template.nii.gz


In [16]:
# Cell 3: BET - Skull Stripping
def bet_skull_strip(input_file, brain_file, mask_file):
    """BET equivalent skull stripping"""
    try:
        img = nib.load(input_file)
        data = img.get_fdata().astype(np.float32)
        
        brain_voxels = data[data > 0]
        if len(brain_voxels) == 0:
            return False
        
        # Multi-threshold
        thresholds = threshold_multiotsu(brain_voxels, classes=3)
        brain_mask = data > (thresholds[0] * 0.4)
        
        # Morphological operations
        brain_mask = morphology.remove_small_objects(brain_mask, min_size=1000)
        brain_mask = ndimage.binary_fill_holes(brain_mask)
        struct = morphology.ball(2)
        brain_mask = morphology.binary_closing(brain_mask, struct)
        brain_mask = morphology.binary_opening(brain_mask, struct)
        
        # Apply mask
        skull_stripped = data * brain_mask
        
        # Save
        brain_img = nib.Nifti1Image(skull_stripped, img.affine, img.header)
        nib.save(brain_img, brain_file)
        mask_img = nib.Nifti1Image(brain_mask.astype(np.uint8), img.affine, img.header)
        nib.save(mask_img, mask_file)
        
        return True
    except Exception as e:
        print(f'❌ BET failed: {e}')
        return False

print('✅ BET function ready')

✅ BET function ready


In [17]:
# Cell 4: FAST - Bias Field Correction
def fast_bias_correct(input_file, output_file, mask_file):
    """FAST equivalent bias field correction"""
    try:
        img = nib.load(input_file)
        data = img.get_fdata().astype(np.float32)
        mask_img = nib.load(mask_file)
        mask = mask_img.get_fdata() > 0
        
        # N4 bias correction
        sitk_img = sitk.GetImageFromArray(data)
        sitk_mask = sitk.GetImageFromArray(mask.astype(np.uint8))
        
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([50, 50, 50, 50])
        corrector.SetConvergenceThreshold(0.001)
        corrector.SetBiasFieldFullWidthAtHalfMaximum(0.15)
        
        corrected = corrector.Execute(sitk_img, sitk_mask)
        corrected_data = sitk.GetArrayFromImage(corrected)
        
        corrected_img = nib.Nifti1Image(corrected_data, img.affine, img.header)
        nib.save(corrected_img, output_file)
        
        return True
    except Exception as e:
        print(f'❌ FAST failed: {e}')
        return False

print('✅ FAST function ready')

✅ FAST function ready


In [18]:
# Cell 5: FLIRT - Registration to MNI
def flirt_register(input_file, output_file, template_path):
    """FLIRT equivalent registration to MNI"""
    try:
        moving = sitk.ReadImage(str(input_file))
        fixed = sitk.ReadImage(str(template_path))
        
        registration = sitk.ImageRegistrationMethod()
        registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
        registration.SetMetricSamplingStrategy(registration.RANDOM)
        registration.SetMetricSamplingPercentage(0.01)
        registration.SetInterpolator(sitk.sitkLinear)
        registration.SetOptimizerAsRegularStepGradientDescent(
            learningRate=1.0, minStep=0.001, numberOfIterations=100)
        registration.SetShrinkFactorsPerLevel([4, 2, 1])
        registration.SetSmoothingSigmasPerLevel([2, 1, 0])
        registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        
        initial_transform = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.AffineTransform(3),
            sitk.CenteredTransformInitializerFilter.GEOMETRY)
        registration.SetInitialTransform(initial_transform, inPlace=False)
        
        final_transform = registration.Execute(fixed, moving)
        registered = sitk.Resample(moving, fixed, final_transform, sitk.sitkLinear, 0.0)
        sitk.WriteImage(registered, str(output_file))
        
        return True
    except Exception as e:
        print(f'❌ FLIRT failed: {e}')
        return False

print('✅ FLIRT function ready')

✅ FLIRT function ready


In [19]:
# Cell 6: Register Mask to MNI
def register_mask_to_mni(mask_file, output_file):
    """Register brain mask to MNI space"""
    try:
        mask_img = nib.load(mask_file)
        mask_data = mask_img.get_fdata().astype(np.float32)
        
        target_shape = (91, 109, 91)
        zoom_factors = [target_shape[i] / mask_data.shape[i] for i in range(3)]
        resampled_mask = ndimage.zoom(mask_data, zoom_factors, order=0)
        
        # Ensure exact shape
        if resampled_mask.shape != target_shape:
            result = resampled_mask
            for i in range(3):
                current_size = result.shape[i]
                target_size = target_shape[i]
                if current_size < target_size:
                    pad_total = target_size - current_size
                    pad_before = pad_total // 2
                    pad_after = pad_total - pad_before
                    pad_width = [(0, 0)] * 3
                    pad_width[i] = (pad_before, pad_after)
                    result = np.pad(result, pad_width, mode='constant', constant_values=0)
                elif current_size > target_size:
                    crop_total = current_size - target_size
                    crop_before = crop_total // 2
                    crop_after = crop_before + target_size
                    slices = [slice(None)] * 3
                    slices[i] = slice(crop_before, crop_after)
                    result = result[tuple(slices)]
            resampled_mask = result
        
        resampled_mask = (resampled_mask > 0.5).astype(np.uint8)
        mni_affine = np.array([[-2.0, 0.0, 0.0, 90.0], [0.0, 2.0, 0.0, -126.0],
                               [0.0, 0.0, 2.0, -72.0], [0.0, 0.0, 0.0, 1.0]])
        mask_img = nib.Nifti1Image(resampled_mask, mni_affine)
        nib.save(mask_img, output_file)
        
        return True
    except Exception as e:
        print(f'❌ Mask registration failed: {e}')
        return False

print('✅ Mask registration function ready')

✅ Mask registration function ready


In [ ]:
# Cell 7 FIXED: Z-score Normalization (ENTIRE VOLUME)
def z_score_normalize_fixed(input_file, output_file, mask_file):
    """Per-person Z-score normalization - ENTIRE VOLUME (FIXED)"""
    try:
        img = nib.load(input_file)
        data = img.get_fdata().astype(np.float32)
        mask_img = nib.load(mask_file)
        mask = mask_img.get_fdata() > 0
        
        if np.sum(mask) > 0:
            brain_voxels = data[mask]
            if len(brain_voxels) > 0:
                mean_val = np.mean(brain_voxels)
                std_val = np.std(brain_voxels)
                if std_val > 0:
                    # FIXED: Normalize ENTIRE volume using brain statistics
                    normalized = (data - mean_val) / std_val
                else:
                    normalized = data.copy()
            else:
                normalized = data.copy()
        else:
            normalized = data.copy()
        
        normalized_img = nib.Nifti1Image(normalized, img.affine, img.header)
        nib.save(normalized_img, output_file)
        
        return True
    except Exception as e:
        print(f'❌ Z-score normalization failed: {e}')
        return False

print('✅ Z-score normalization function ready (FIXED - normalizes entire volume)')

In [21]:
# Cell 8: Verify Files
def verify_files(files):
    """Verify quality of each file"""
    verification = {'overall_passed': True, 'file_results': {}, 
                   'summary': {'total': 0, 'passed': 0, 'failed': 0}}
    
    for file_type, file_path in files.items():
        result = {'passed': True, 'issues': [], 'metrics': {}}
        try:
            if not file_path.exists():
                result['passed'] = False
                result['issues'].append('File missing')
                verification['overall_passed'] = False
                continue
            
            img = nib.load(file_path)
            data = img.get_fdata()
            
            if np.any(np.isnan(data)):
                result['passed'] = False
                result['issues'].append('Contains NaN')
                verification['overall_passed'] = False
            
            if np.any(np.isinf(data)):
                result['passed'] = False
                result['issues'].append('Contains Inf')
                verification['overall_passed'] = False
            
            result['metrics']['shape'] = data.shape
            if file_type == 'mni_registered' and data.shape != (91, 109, 91):
                result['passed'] = False
                result['issues'].append(f'Wrong MNI shape: {data.shape}')
                verification['overall_passed'] = False
            
            brain_voxels = np.sum(data > 0)
            result['metrics']['brain_voxels'] = int(brain_voxels)
            
            if brain_voxels < 1000 and file_type != 'brain_mask':
                result['passed'] = False
                result['issues'].append('Too few brain voxels')
                verification['overall_passed'] = False
                
        except Exception as e:
            result['passed'] = False
            result['issues'].append(f'Error: {str(e)}')
            verification['overall_passed'] = False
        
        verification['file_results'][file_type] = result
        verification['summary']['total'] += 1
        if result['passed']:
            verification['summary']['passed'] += 1
        else:
            verification['summary']['failed'] += 1
    
    return verification

print('✅ Verification function ready')

✅ Verification function ready


In [22]:
# Cell 9: Calculate Quality Metrics
def calculate_quality(input_file, output_file, mask_file, verification):
    """Calculate quality metrics"""
    quality = {'fsl_pipeline_followed': True, 
               'verification_passed': verification['overall_passed'],
               'verification_summary': verification['summary']}
    
    try:
        original = nib.load(input_file).get_fdata()
        final = nib.load(output_file).get_fdata()
        mask = nib.load(mask_file).get_fdata() > 0
        
        quality['original_shape'] = original.shape
        quality['final_shape'] = final.shape
        quality['mni_shape_correct'] = final.shape == (91, 109, 91)
        
        brain_voxels = np.sum(mask)
        quality['brain_voxels'] = int(brain_voxels)
        quality['brain_volume_ml'] = float(brain_voxels * 8 / 1000)
        
        if brain_voxels > 0:
            brain_intensities = final[mask]
            quality['final_mean'] = float(np.mean(brain_intensities))
            quality['final_std'] = float(np.std(brain_intensities))
            z_score_good = abs(quality['final_mean']) < 0.1 and abs(quality['final_std'] - 1.0) < 0.1
            quality['z_score_quality'] = z_score_good
        else:
            quality['final_mean'] = 0.0
            quality['final_std'] = 0.0
            quality['z_score_quality'] = False
        
        components = [
            1.0 if quality['mni_shape_correct'] else 0.0,
            1.0 if verification['overall_passed'] else 0.0,
            min(quality['brain_volume_ml'] / 1500.0, 1.0),
            1.0 if quality['z_score_quality'] else 0.0
        ]
        quality['overall_score'] = float(np.mean(components))
        
        if quality['overall_score'] >= 0.9:
            quality['grade'] = 'Excellent'
        elif quality['overall_score'] >= 0.8:
            quality['grade'] = 'Good'
        elif quality['overall_score'] >= 0.7:
            quality['grade'] = 'Fair'
        else:
            quality['grade'] = 'Poor'
            
    except Exception as e:
        quality['overall_score'] = 0.0
        quality['grade'] = 'Failed'
        quality['error'] = str(e)
    
    return quality

print('✅ Quality calculation function ready')

✅ Quality calculation function ready


In [23]:
# Cell 10: Process Single Subject
def process_subject(subject_id, session, input_dir, output_dir, mni_template):
    """Process single subject through complete FSL pipeline"""
    start_time = time.time()
    
    # Create output directory
    subj_output = Path(output_dir) / subject_id / session
    subj_output.mkdir(parents=True, exist_ok=True)
    
    # Find input file
    anat_dir = Path(input_dir) / subject_id / session / 'anat'
    if not anat_dir.exists():
        return {'success': False, 'error': 'Anat directory not found'}
    
    t1w_files = list(anat_dir.glob('*T1w*.nii.gz'))
    if not t1w_files:
        unit1_files = list(anat_dir.glob('*UNIT1*.nii.gz'))
        if not unit1_files:
            return {'success': False, 'error': 'No input file found'}
        input_file = unit1_files[0]
    else:
        input_file = t1w_files[0]
    
    try:
        # Step 1: BET
        bet_brain = subj_output / f'{subject_id}_{session}_brain.nii.gz'
        bet_mask = subj_output / f'{subject_id}_{session}_brain_mask.nii.gz'
        if not bet_skull_strip(input_file, bet_brain, bet_mask):
            return {'success': False, 'error': 'BET failed'}
        
        # Step 2: FAST
        fast_restore = subj_output / f'{subject_id}_{session}_restore.nii.gz'
        if not fast_bias_correct(bet_brain, fast_restore, bet_mask):
            return {'success': False, 'error': 'FAST failed'}
        
        # Step 3: FLIRT
        flirt_mni = subj_output / f'{subject_id}_{session}_mni.nii.gz'
        if not flirt_register(fast_restore, flirt_mni, mni_template):
            return {'success': False, 'error': 'FLIRT failed'}
        
        # Register mask
        flirt_mask = subj_output / f'{subject_id}_{session}_mni_mask.nii.gz'
        if not register_mask_to_mni(bet_mask, flirt_mask):
            return {'success': False, 'error': 'Mask registration failed'}
        
        # Step 4: Verify
        verification = verify_files({
            'input': input_file, 'skull_stripped': bet_brain,
            'bias_corrected': fast_restore, 'mni_registered': flirt_mni,
            'brain_mask': bet_mask
        })
        
        # Step 5: Z-score
        final_output = subj_output / f'{subject_id}_{session}_preprocessed.nii.gz'
        if not z_score_normalize(flirt_mni, final_output, flirt_mask):
            return {'success': False, 'error': 'Z-score failed'}
        
        # Calculate quality
        quality = calculate_quality(input_file, final_output, flirt_mask, verification)
        
        processing_time = time.time() - start_time
        
        return {
            'success': True,
            'subject_id': subject_id,
            'session': session,
            'processing_time': processing_time,
            'quality_metrics': quality,
            'verification_results': verification
        }
        
    except Exception as e:
        return {
            'success': False,
            'subject_id': subject_id,
            'session': session,
            'error': str(e),
            'processing_time': time.time() - start_time
        }

print('✅ Process subject function ready')

✅ Process subject function ready


In [24]:
# Cell 11: Find All Subjects (SESSION 1 ONLY)
input_dir = 'Mri'
output_dir = 'outputs/standalone'

subjects = []
for subject_dir in Path(input_dir).glob('sub-*'):
    if subject_dir.is_dir():
        # ONLY PROCESS ses-01
        session_dir = subject_dir / 'ses-01'
        if session_dir.exists() and (session_dir / 'anat').exists():
            subjects.append((subject_dir.name, 'ses-01'))

print(f'🔍 Found {len(subjects)} subjects to process (ses-01 only)')
print(f'📋 First few: {subjects[:5]}')

🔍 Found 417 subjects to process (ses-01 only)
📋 First few: [('sub-0001', 'ses-01'), ('sub-0002', 'ses-01'), ('sub-0003', 'ses-01'), ('sub-0004', 'ses-01'), ('sub-0005', 'ses-01')]


In [25]:
# Cell 12: Process ALL Subjects (AUTO-RESUME)
print('🚀 Processing ALL MRI subjects (ses-01 only)...')
print('=' * 60)

results = []
successful = 0
failed = 0
skipped = 0
start_time = time.time()

for subject_id, session in tqdm(subjects, desc='Processing MRI'):
    
    # CHECK IF ALREADY PROCESSED - AUTO SKIP
    final_file = Path(output_dir) / subject_id / session / f'{subject_id}_{session}_preprocessed.nii.gz'
    
    if final_file.exists():
        tqdm.write(f'\n⏭️  Skipping: {subject_id} {session} (already processed)')
        skipped += 1
        successful += 1  # Count as successful
        continue
    
    # Process if not done yet
    tqdm.write(f'\n🔄 Processing: {subject_id} {session}')
    
    result = process_subject(subject_id, session, input_dir, output_dir, mni_template)
    results.append(result)
    
    if result['success']:
        successful += 1
        quality = result['quality_metrics']['overall_score']
        grade = result['quality_metrics']['grade']
        tqdm.write(f'   ✅ Success! Quality: {quality:.3f} ({grade})')
    else:
        failed += 1
        tqdm.write(f'   ❌ Failed: {result["error"]}')

total_time = time.time() - start_time
print('\n' + '=' * 60)
print('🎉 PROCESSING COMPLETE!')
print('=' * 60)
print(f'✅ Successful: {successful}/{len(subjects)}')
print(f'⏭️  Skipped (already done): {skipped}/{len(subjects)}')
print(f'❌ Failed: {failed}/{len(subjects)}')
print(f'📈 Success Rate: {successful/len(subjects)*100:.1f}%')
print(f'⏰ Total Time: {total_time/60:.1f} minutes')
if len(subjects) - skipped > 0:
    print(f'⚡ Avg Time per Subject: {total_time/(len(subjects)-skipped):.1f} seconds')

🚀 Processing ALL MRI subjects (ses-01 only)...


Processing MRI:   0%|          | 0/417 [00:00<?, ?it/s]


⏭️  Skipping: sub-0001 ses-01 (already processed)

⏭️  Skipping: sub-0002 ses-01 (already processed)

⏭️  Skipping: sub-0003 ses-01 (already processed)

⏭️  Skipping: sub-0004 ses-01 (already processed)

⏭️  Skipping: sub-0005 ses-01 (already processed)

⏭️  Skipping: sub-0006 ses-01 (already processed)

⏭️  Skipping: sub-0007 ses-01 (already processed)

⏭️  Skipping: sub-0008 ses-01 (already processed)

⏭️  Skipping: sub-0009 ses-01 (already processed)

⏭️  Skipping: sub-0010 ses-01 (already processed)

⏭️  Skipping: sub-0011 ses-01 (already processed)

⏭️  Skipping: sub-0012 ses-01 (already processed)

⏭️  Skipping: sub-0013 ses-01 (already processed)

⏭️  Skipping: sub-0014 ses-01 (already processed)

⏭️  Skipping: sub-0015 ses-01 (already processed)

⏭️  Skipping: sub-0016 ses-01 (already processed)

⏭️  Skipping: sub-0017 ses-01 (already processed)

⏭️  Skipping: sub-0018 ses-01 (already processed)

⏭️  Skipping: sub-0019 ses-01 (already processed)

⏭️  Skipping: sub-0020 ses-01 

In [26]:
# Cell 13: Results Summary
successful_results = [r for r in results if r['success']]

if successful_results:
    data = []
    for r in successful_results:
        qm = r['quality_metrics']
        data.append({
            'Subject': r['subject_id'],
            'Session': r['session'],
            'Quality': qm['overall_score'],
            'Grade': qm['grade'],
            'Brain Volume (mL)': qm['brain_volume_ml'],
            'Time (s)': r['processing_time']
        })
    
    df = pd.DataFrame(data)
    print('\n📊 Quality Summary:')
    print(df.describe())
    print('\n🏆 Grade Distribution:')
    print(df['Grade'].value_counts())
    
    csv_path = Path(output_dir) / 'all_results.csv'
    df.to_csv(csv_path, index=False)
    print(f'\n💾 Results saved to: {csv_path}')
    
    display(df.head(10))
else:
    print('❌ No successful results to analyze')

❌ No successful results to analyze
